# 4. Math and Statistics

Most math in NumPy is **element-wise** and **vectorized**: you write the operation once and it's applied to every element, without a Python loop.

In [1]:
import numpy as np

rng = np.random.default_rng(0)
arr = np.arange(1, 13).reshape(3, 4)
arr

array([[ 1,  2,  3,  4],
       [ 5,  6,  7,  8],
       [ 9, 10, 11, 12]])

## A note on `axis` (read this first)

Reductions like `sum`, `mean` and `max` take an `axis` argument. The easiest way to think about it: **`axis` is the dimension that gets collapsed**.

- `axis=0` collapses the rows, so you get **one result per column**
- `axis=1` collapses the columns, so you get **one result per row**
- no axis means reduce over everything and get a single number

For our 3×4 array, `axis=0` gives 4 values and `axis=1` gives 3.

In [2]:
print('per column (axis=0):', arr.sum(axis=0))
print('per row    (axis=1):', arr.sum(axis=1))
print('everything:         ', arr.sum())

per column (axis=0): [15 18 21 24]
per row    (axis=1): [10 26 42]
everything:          78


Pass `keepdims=True` to keep the reduced axis around with length 1. That makes it easy to broadcast the result back against the original, e.g. to get each value's share of its row:

In [3]:
row_totals = arr.sum(axis=1, keepdims=True)
print(row_totals.shape)
print(np.round(arr / row_totals, 2))

(3, 1)
[[0.1  0.2  0.3  0.4 ]
 [0.19 0.23 0.27 0.31]
 [0.21 0.24 0.26 0.29]]


## Arithmetic

The usual operators work element-wise, both with scalars and with arrays of the same shape.

In [4]:
print(arr + 10)
print(arr * 2)
print(arr / 4)
print(arr // 4)      # floor division
print(arr % 4)       # remainder
print(arr ** 2)

[[11 12 13 14]
 [15 16 17 18]
 [19 20 21 22]]
[[ 2  4  6  8]
 [10 12 14 16]
 [18 20 22 24]]
[[0.25 0.5  0.75 1.  ]
 [1.25 1.5  1.75 2.  ]
 [2.25 2.5  2.75 3.  ]]
[[0 0 0 1]
 [1 1 1 2]
 [2 2 2 3]]
[[1 2 3 0]
 [1 2 3 0]
 [1 2 3 0]]
[[  1   4   9  16]
 [ 25  36  49  64]
 [ 81 100 121 144]]


In [5]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
print(a + b, a * b, b / a)

[11 22 33] [10 40 90] [10. 10. 10.]


Every operator has a function version (`np.add`, `np.multiply`, ...). The functions accept an `out` argument, so you can write the result into an existing array instead of allocating a new one. The augmented operators (`+=`, `*=`) do the same thing.

In [6]:
a = np.array([1, 2, 3])
np.add(a, [8, 9, 10], out=a)
print(a)

a *= 2
print(a)

[ 9 11 13]
[18 22 26]


## Exponents, logs, roots

In [7]:
x = np.array([1, 2, 4, 8])
print(np.exp(x))
print(np.log(x))      # natural log
print(np.log2(x))
print(np.log10(x))
print(np.sqrt(x))
print(np.power(x, 3))

[2.71828183e+00 7.38905610e+00 5.45981500e+01 2.98095799e+03]
[0.         0.69314718 1.38629436 2.07944154]
[0. 1. 2. 3.]
[0.         0.30103    0.60205999 0.90308999]
[1.         1.41421356 2.         2.82842712]
[  1   8  64 512]


`log` of zero or a negative number gives `-inf` or `nan` with a warning, not an error. If you're working with small probabilities, `np.log1p` and `np.expm1` are more accurate near zero.

In [8]:
with np.errstate(divide='ignore', invalid='ignore'):
    print(np.log(np.array([0.0, -1.0])))
print(np.log1p(1e-10), np.log(1 + 1e-10))

[-inf  nan]
9.999999999500001e-11 1.000000082690371e-10


## Trigonometry

Angles are in radians. Use `np.deg2rad` / `np.rad2deg` to convert.

In [9]:
angles = np.deg2rad([0, 30, 45, 90])
print(np.round(np.sin(angles), 3))
print(np.round(np.cos(angles), 3))
print(np.pi, np.e)

[0.    0.5   0.707 1.   ]
[1.    0.866 0.707 0.   ]
3.141592653589793 2.718281828459045


## Rounding, absolute values and clipping

In [10]:
x = np.array([-2.5, -1.5, -0.4, 0.5, 1.5, 2.567])
print(np.round(x))        # rounds half to even: 0.5 -> 0, 1.5 -> 2
print(np.round(x, 1))
print(np.floor(x))
print(np.ceil(x))
print(np.trunc(x))
print(np.abs(x))
print(np.clip(x, -1, 1))  # limit values to a range

[-2. -2. -0.  0.  2.  3.]
[-2.5 -1.5 -0.4  0.5  1.5  2.6]
[-3. -2. -1.  0.  1.  2.]
[-2. -1. -0.  1.  2.  3.]
[-2. -1. -0.  0.  1.  2.]
[2.5   1.5   0.4   0.5   1.5   2.567]
[-1.  -1.  -0.4  0.5  1.   1. ]


`np.modf` splits floats into their fractional and integer parts:

In [11]:
frac, whole = np.modf(np.array([3.75, -1.25, 10.5]))
print('fractional:', frac)
print('integer:   ', whole)

fractional: [ 0.75 -0.25  0.5 ]
integer:    [ 3. -1. 10.]


## Min, max and where they are

In [12]:
m = rng.integers(0, 100, size=(3, 4))
print(m)
print('max per column:', m.max(axis=0))
print('min per row:   ', m.min(axis=1))
print('overall max:   ', m.max())

[[85 63 51 26]
 [30  4  7  1]
 [17 81 64 91]]
max per column: [85 81 64 91]
min per row:    [26  1 17]
overall max:    91


`argmin` / `argmax` return the **position** instead of the value. With no axis, the index refers to the flattened array, and `np.unravel_index` converts it back to a row/column pair.

In [13]:
print('row of the max in each column:', m.argmax(axis=0))
print('column of the min in each row:', m.argmin(axis=1))

flat = m.argmax()
print('flat index:', flat, '-> (row, col):', np.unravel_index(flat, m.shape))

row of the max in each column: [0 2 2 2]
column of the min in each row: [3 3 0]
flat index: 11 -> (row, col): (np.int64(2), np.int64(3))


`np.maximum` / `np.minimum` compare two arrays element by element. Not the same as `max`/`min`, which reduce one array.

In [14]:
a = np.array([1, 3, 5, 7, 9])
b = np.array([0, 4, 3, 8, 7])
print(np.maximum(a, b))
print(np.minimum(a, b))
print(np.maximum(a, 4))   # handy for things like ReLU

[1 4 5 8 9]
[0 3 3 7 7]
[4 4 5 7 9]


## Descriptive statistics

In [15]:
scores = rng.normal(70, 10, size=(5, 3)).round(1)   # 5 students, 3 exams
scores

array([[83. , 79.5, 63. ],
       [57.3, 63.8, 70.4],
       [46.7, 67.8, 57.5],
       [62.7, 64.6, 66.8],
       [74.1, 80.4, 68.7]])

In [16]:
print('mean per exam:   ', scores.mean(axis=0).round(2))
print('median per exam: ', np.median(scores, axis=0))
print('std per exam:    ', scores.std(axis=0).round(2))
print('var per exam:    ', scores.var(axis=0).round(2))
print('mean per student:', scores.mean(axis=1).round(2))

mean per exam:    [64.76 71.22 65.28]
median per exam:  [62.7 67.8 66.8]
std per exam:     [12.7   7.26  4.6 ]
var per exam:     [161.2   52.68  21.19]
mean per student: [75.17 63.83 57.33 64.7  74.4 ]


Note that `std` and `var` default to the **population** formula (dividing by N). For the sample version (N-1), which is what most stats packages use, pass `ddof=1`:

In [17]:
x = np.array([2, 4, 4, 4, 5, 5, 7, 9])
print(x.std(), x.std(ddof=1))

2.0 2.138089935299395


In [18]:
x = rng.random(1000)
print(np.percentile(x, [5, 50, 95]))
print(np.quantile(x, [0.05, 0.5, 0.95]))   # same thing, but 0-1 instead of 0-100

[0.06620677 0.52609514 0.9452206 ]
[0.06620677 0.52609514 0.9452206 ]


A weighted average:

In [19]:
grades = np.array([80, 90, 70])
weights = np.array([0.2, 0.5, 0.3])
np.average(grades, weights=weights)

np.float64(82.0)

## Covariance and correlation

`np.cov` and `np.corrcoef` treat each **row** as a variable by default. If your variables are in columns (the usual layout for data), pass `rowvar=False`.

In [20]:
height = rng.normal(170, 10, 200)
weight = 0.5 * height + rng.normal(0, 5, 200)
data = np.column_stack([height, weight])    # shape (200, 2), variables in columns

print(np.cov(data, rowvar=False).round(1))
print(np.corrcoef(data, rowvar=False).round(3))

[[95.9 54.6]
 [54.6 59.3]]
[[1.    0.724]
 [0.724 1.   ]]


## Cumulative sums and products

In [21]:
a = np.arange(1, 7).reshape(2, 3)
print(a)
print(np.cumsum(a))            # flattened
print(np.cumsum(a, axis=0))    # running total down each column
print(np.cumsum(a, axis=1))    # running total along each row
print(np.cumprod(a, axis=1))

[[1 2 3]
 [4 5 6]]
[ 1  3  6 10 15 21]
[[1 2 3]
 [5 7 9]]
[[ 1  3  6]
 [ 4  9 15]]
[[  1   2   6]
 [  4  20 120]]


`np.diff` goes the other way, giving the differences between neighbours:

In [22]:
prices = np.array([100, 102, 101, 105, 110])
print(np.diff(prices))
print(np.diff(prices) / prices[:-1])   # daily returns

[ 2 -1  4  5]
[ 0.02       -0.00980392  0.03960396  0.04761905]


## Comparisons and boolean logic

Comparisons give boolean arrays. Combine them with `&`, `|` and `~` (not `and`/`or`/`not`), and **always wrap each condition in parentheses** because `&` binds tighter than `<`.

In [23]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([5, 4, 3, 2, 1])
print(a == b)
print(a < 3)
print((a > 1) & (a < 5))
print((a < 2) | (a > 4))
print(~(a > 2))

[False False  True False False]
[ True  True False False False]
[False  True  True  True False]
[ True False False False  True]
[ True  True False False False]


`True` counts as 1, so `sum` counts matches and `mean` gives the fraction:

In [24]:
x = rng.normal(size=1000)
print('values above 1:', (x > 1).sum())
print('fraction above 1:', (x > 1).mean())
print('any negative?', (x < 0).any(), ' all finite?', np.isfinite(x).all())

values above 1: 156
fraction above 1: 0.156
any negative? True  all finite? True


Never compare floats with `==`. Use `np.isclose` / `np.allclose`:

In [25]:
print(0.1 + 0.2 == 0.3)
print(np.isclose(0.1 + 0.2, 0.3))
print(np.allclose([1.0, 2.0], [1.0, 2.0 + 1e-10]))

False
True
True
